# HistGradientBoostingRegressor Modelo Original

In [3]:
import numpy as np  # type: ignore
import pandas as pd 
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
import plotly.express as px
import plotly.graph_objects as go
from sklearn.linear_model import Ridge

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))


ruta = r'C:\Users\jthow\iCloudDrive\Documents\3_Maestria_Estadistica_UNINORTE\3_Tercer_Semestre\Machine_Learning\tornados.csv.zip'  
df = pd.read_csv(ruta)  
df['loss'] = df['loss'].replace(0, pd.NA)
df['loss'] = df['loss'].interpolate(method='linear')
df['mag'] = df['mag'].fillna(df['mag'].mean())
df.isnull().sum()

om              0
yr              0
mo              0
dy              0
date            0
time            0
tz              0
datetime_utc    0
st              0
stf             0
mag             0
inj             0
fat             0
loss            0
slat            0
slon            0
elat            0
elon            0
len             0
wid             0
ns              0
sn              0
f1              0
f2              0
f3              0
f4              0
fc              0
dtype: int64

In [5]:
# -------------------------------
# Paso 1: Importar librerías
# -------------------------------
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.inspection import permutation_importance
from joblib import dump

# -------------------------------
# Paso 2: Definir X y y
# -------------------------------
# Asegúrate de que tu DataFrame `df` ya esté cargado antes de ejecutar esto
X = df[['mag', 'slat', 'slon', 'elat', 'elon', 'len', 'wid', 'f1', 'f2', 'f3', 'f4', 'loss']]
y = df['inj']

# -------------------------------
# Paso 3: Dividir los datos
# -------------------------------
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# -------------------------------
# Paso 4: Entrenar el modelo
# -------------------------------
hgb = HistGradientBoostingRegressor()
hgb.fit(X_train, y_train)

# -------------------------------
# Paso 5: Evaluar el modelo
# -------------------------------
print("Training set score: {:.2f}".format(hgb.score(X_train, y_train)))
print("Test set score: {:.2f}".format(hgb.score(X_test, y_test)))

# -------------------------------
# Paso 6: Calcular importancia de características
# -------------------------------
result = permutation_importance(hgb, X_test, y_test, n_repeats=10, random_state=42)
coeficientes = pd.Series(result.importances_mean, index=X_train.columns)

print("\nImportancia de las características del modelo HistGradientBoosting:")
print(coeficientes.sort_values(ascending=False))

# -------------------------------
# Paso 7: Guardar el modelo entrenado
# -------------------------------
dump(hgb, 'histgradientboosting_model.joblib')
print("\nModelo guardado como 'histgradientboosting_model.joblib'")


Training set score: 0.50
Test set score: 0.29

Importancia de las características del modelo HistGradientBoosting:
mag     0.280667
loss    0.102649
slon    0.016754
wid     0.006349
elat    0.005087
elon    0.002508
f3     -0.000154
f4     -0.001054
f1     -0.001518
f2     -0.001574
len    -0.002388
slat   -0.013630
dtype: float64

Modelo guardado como 'histgradientboosting_model.joblib'


## Metricas para el Modelo Original de Regresión

In [7]:
# ------------------------
# Paso 1: Importar los paquetes necesarios
# ------------------------
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_percentage_error, mean_squared_error, r2_score
from sklearn.ensemble import HistGradientBoostingRegressor
from statsmodels.stats.diagnostic import acorr_ljungbox
from scipy.stats import jarque_bera
from joblib import dump  # Importar para guardar el modelo
from time import time  # Importación de time

# ------------------------
# Paso 2: Definir y entrenar el modelo HistGradientBoostingRegressor
# ------------------------
# Instanciar el modelo
grid_hgb = HistGradientBoostingRegressor()

# Medir el tiempo de entrenamiento
start_time = time()
grid_hgb.fit(X_train, y_train)
training_time_hgb = time() - start_time

# Guardar el modelo entrenado
dump(grid_hgb, 'grid_hgb.joblib')

# ------------------------
# Paso 3: Hacer predicciones
# ------------------------
# Predicciones con el modelo entrenado
y_pred_hgb = grid_hgb.predict(X_test)

# ------------------------
# Paso 4: Calcular las métricas
# ------------------------
mape_hgb = mean_absolute_percentage_error(y_test, y_pred_hgb)
rmse_hgb = np.sqrt(mean_squared_error(y_test, y_pred_hgb))
r2_hgb = r2_score(y_test, y_pred_hgb)

# Ljung-Box p-value
lb_test_hgb = acorr_ljungbox(y_test - y_pred_hgb, lags=[10])
lb_p_value_hgb = lb_test_hgb['lb_pvalue'].iloc[0]  # Valor p para lag 10

# Jarque-Bera p-value
jb_p_value_hgb = jarque_bera(y_test - y_pred_hgb)[1]

# ------------------------
# Paso 5: Crear DataFrame con los resultados
# ------------------------
resultados_hgb = pd.DataFrame({
    'Modelo': ['HistGradientBoosting'],
    'MAPE': [f"{mape_hgb:.2f}"],
    'RMSE': [f"{rmse_hgb:.2f}"],
    'R Cuadrado': [f"{r2_hgb:.2f}"],
    'Ljung-Box Test p-value': [f"{lb_p_value_hgb:.4f}"],
    'Jarque-Bera p-value': [f"{jb_p_value_hgb:.4f}"],
    'CPU time (s)': [round(training_time_hgb, 2)]
})

# ------------------------
# Paso 6: Mostrar los resultados
# ------------------------
print("Métricas HistGradientBoostingRegressor:")
display(resultados_hgb)


Métricas HistGradientBoostingRegressor:


,Modelo,MAPE,RMSE,R Cuadrado,Ljung-Box Test p-value,Jarque-Bera p-value,CPU time (s)
0,HistGradientBoosting,2218089056950298.75,19.37,0.29,0.9349,0.0000,1.39


## Optimizado

In [10]:
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import GridSearchCV, KFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    mean_absolute_percentage_error, 
    mean_squared_error, 
    r2_score,
    mean_absolute_error
)
from statsmodels.stats.diagnostic import acorr_ljungbox
from scipy.stats import jarque_bera
from joblib import dump
from time import time

def train_hist_gradient_boosting_regressor(X_train, X_test, y_train, y_test):
    """
    Entrenar y evaluar un HistGradientBoostingRegressor con GridSearchCV.
    
    Parámetros:
    X_train, X_test: Features de entrenamiento y prueba
    y_train, y_test: Etiquetas de entrenamiento y prueba
    
    Retorna:
    DataFrame con métricas de rendimiento del modelo
    """
    # Definir pipeline con escalado
    pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('model', HistGradientBoostingRegressor(random_state=42))
    ])
    
    # Grid de hiperparámetros exhaustivo y eficiente
    param_grid = {
        'model__max_iter': [100, 200, 300],
        'model__max_depth': [None, 10, 20],
        'model__learning_rate': [0.01, 0.05, 0.1],
        'model__min_samples_leaf': [20, 30],
        'model__l2_regularization': [0, 0.1]
    }
    
    # Configuración de cross-validation
    cv = KFold(n_splits=5, shuffle=True, random_state=42)
    
    # Medir tiempo de entrenamiento
    start_time = time()
    
    # Realizar GridSearchCV
    grid_search = GridSearchCV(
        pipeline, 
        param_grid, 
        cv=cv, 
        n_jobs=-1, 
        scoring='neg_mean_squared_error',
        verbose=0
    )
    grid_search.fit(X_train, y_train)
    
    # Calcular tiempo de entrenamiento
    training_time = time() - start_time
    
    # Obtener mejor modelo
    best_model = grid_search.best_estimator_
    
    # Guardar modelo
    dump(grid_search, 'grid_hgb_optimized.joblib')
    
    # Predicciones
    y_pred = best_model.predict(X_test)
    
    # Calcular métricas
    try:
        # Calcular métricas de regresión
        mape = mean_absolute_percentage_error(y_test, y_pred)
        rmse = np.sqrt(mean_squared_error(y_test, y_pred))
        r2 = r2_score(y_test, y_pred)
        
        # Calcular residuos
        residuos = y_test - y_pred
        
        # Ljung-Box p-value
        try:
            lb_test = acorr_ljungbox(residuos, lags=[10])
            lb_p_value = lb_test['lb_pvalue'].iloc[0]
        except Exception:
            lb_p_value = None
        
        # Jarque-Bera p-value
        try:
            jb_p_value = jarque_bera(residuos)[1]
        except Exception:
            jb_p_value = None
        
        # Crear DataFrame de resultados
        resultados_hgb = pd.DataFrame({
            'Modelo': ['HistGradientBoosting optimizado'],
            'MAPE': [f"{mape:.2f}"],
            'RMSE': [f"{rmse:.2f}"],
            'R Cuadrado': [f"{r2:.2f}"],
            'Ljung-Box Test p-value': [f"{lb_p_value:.4f}" if lb_p_value is not None else "N/A"],
            'Jarque-Bera p-value': [f"{jb_p_value:.4f}" if jb_p_value is not None else "N/A"],
            'CPU time (s)': [round(training_time, 2)]
        })
        
        return resultados_hgb
    
    except Exception as e:
        print(f"Error al calcular métricas: {e}")
        return None

# Uso del método (asumiendo que X_train, X_test, y_train, y_test están definidos)
resultados_hgb = train_hist_gradient_boosting_regressor(X_train, X_test, y_train, y_test)

# Mostrar resultados
print("Métricas HistGradientBoostingRegressor:")
display(resultados_hgb)

Métricas HistGradientBoostingRegressor:


,Modelo,MAPE,RMSE,R Cuadrado,Ljung-Box Test p-value,Jarque-Bera p-value,CPU time (s)
0,HistGradientBoosting optimizado,2149779610850683.25,19.17,0.31,0.9674,0.0000,96.86
